In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline
import matplotlib.pyplot as plt
import sigpy as sp
import pandas as pd
import sigpy.plot as pl
import numpy as np
import os
ksp = np.load(r"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Sigpytutorial/projection_ksp.npy")
coord = np.load(r"/Users/ayman/Library/CloudStorage/OneDrive-King'sCollegeLondon/Project/Sigpytutorial/projection_coord.npy")

import sys
from pathlib import Path

project_root = Path.cwd().parents[1]
sys.path.append(str(project_root))

import aymansigmri as asm

In [ ]:
width=4
gap = 2
num_iters = 50

resize_wid=1024
rw_div2 = int(resize_wid/2)

inner_wid=100
indiv2 = int(inner_wid/2)

In [ ]:
zte_radial_kspacedlt, zte_radial_coordsdlt, zte_radial_kspacezero, zte_radial_coordszero, mask = asm.generate_zte_data(ksp=ksp, coord=coord, n_missing=(gap), undersampling_factor=1)


cartesian_zte = asm.nufft_gridding(kspace=zte_radial_kspacezero, coords=zte_radial_coordszero)

cartesian_kspace_enl = asm.zero_padding(cart_kspace=cartesian_zte, resize_x=resize_wid, resize_y=resize_wid)

inner_region, inner_mask, start, end = asm.inner_portion(enlarged_kspace=cartesian_kspace_enl, inner_sidelen=inner_wid)

In [ ]:
def zero_padding2d(cart_kspace,resize_x, resize_y):
    image_grid = sp.ifft(cart_kspace)
    enlarged_image_grid = sp.resize(image_grid, [resize_x, resize_y])
    enlarged_cartesian_kspace = sp.fft(enlarged_image_grid, axes=(-2, -1))
    return enlarged_cartesian_kspace


def inner_portion2d(enlarged_kspace, inner_sidelen):
    cx, cy = enlarged_kspace.shape[0] // 2, enlarged_kspace.shape[1] // 2
    N = inner_sidelen/2
    start, end = int(cx-N), int(cx+N)
    isolation_mask = np.ones(enlarged_kspace.shape, dtype=int)
    isolation_mask[start:end, start:end] = 0
    isolated_kspace = enlarged_kspace[start:end, start:end]
    
    return(isolated_kspace, isolation_mask, start, end)

def sampling_density(coords, grid_shape, resize_wid, inner_wid):
    ones = np.ones(coords.shape[:-1], dtype=np.complex64)
    dcf = np.sqrt(coords[..., 0]**2 + coords[..., 1]**2)
    img = sp.nufft_adjoint(ones, coords, oshape=grid_shape)
    kspzer = np.abs(sp.fft(img, axes=tuple(range(-len(grid_shape), 0))))
    enl_ksp = np.abs(zero_padding2d(cart_kspace=kspzer, resize_x=resize_wid, resize_y=resize_wid))
    inner_enl, inner_mask, start, end = inner_portion2d(enlarged_kspace=enl_ksp, inner_sidelen=inner_wid)
    return kspzer, np.abs(inner_enl)


In [ ]:
zte_radial_coordsdlt.shape

In [ ]:

x, x_enl = sampling_density(zte_radial_coordsdlt, (256,256), 512, 100)
xa, xa_enl = sampling_density(zte_radial_coordszero, (256,256), 512, 100)

fig, axes = plt.subplots(2, 3, figsize=(25, 8))

for a, data, title in zip(axes[0, :2], [x, xa], ['deleted', 'zeroed']):
    im = a.imshow(data, cmap='viridis')
    a.set_title(title)
    a.set_aspect('equal', adjustable='box')
    a.set_xlabel('$k_x$')
    a.set_ylabel('$k_y$')
    a.set_xlim(120, 136)
    a.set_ylim(120, 136)
    fig.colorbar(im, ax=a, label='gridded sample density')


for a_enl, data_enl, title_enl in zip(axes[1, :2], [x_enl, xa_enl], ['deleted_enl', 'zeroed_enl']):
    im_enl = a_enl.imshow(data_enl, cmap='viridis')
    a_enl.set_title(title_enl)
    a_enl.set_aspect('equal', adjustable='box')
    a_enl.set_xlabel('$k_x$')
    a_enl.set_ylabel('$k_y$')
    a_enl.set_xlim(0, 100)
    a_enl.set_ylim(0, 100)
    fig.colorbar(im_enl, ax=a_enl, label='gridded sample density')


d = x - xa
v = np.abs(d).max()
im = axes[0, 2].imshow(d, cmap='RdBu_r', vmin=-v, vmax=v)
axes[0, 2].set_title('difference')
axes[0, 2].set_aspect('equal', adjustable='box')
axes[0, 2].set_xlabel('$k_x$')
axes[0, 2].set_ylabel('$k_y$')
axes[0, 2].set_xlim(120, 136)
axes[0, 2].set_ylim(120, 136)
fig.colorbar(im, ax=axes[0, 2], label='difference')




d_enl = x_enl - xa_enl

v_enl = np.abs(d_enl).max()
im_enl = axes[1, 2].imshow(d_enl, cmap='RdBu_r', vmin=-v_enl, vmax=v_enl)
axes[1, 2].set_title('difference_enl')
axes[1, 2].set_aspect('equal', adjustable='box')
axes[1, 2].set_xlabel('$k_x$')
axes[1, 2].set_ylabel('$k_y$')
axes[1, 2].set_xlim(0, 100)
axes[1, 2].set_ylim(0, 100)
fig.colorbar(im_enl, ax=axes[1, 2], label='difference')

plt.tight_layout()
plt.show()

In [ ]:
np.std(xa_enl)

In [ ]:
resizewid = 100
halfresizewid = resizewid//2

legnth=20


fig, axes = plt.subplots(1, 2, figsize=(15, 8))



d = x - xa
mask = np.abs(d) < (np.mean(d) + 2*np.std(d))
mask = mask.astype(float) 
v = np.abs(d).max()
im = axes[0].imshow(mask, cmap='RdBu_r', vmin=0, vmax=1)
axes[0].set_title('difference')
axes[0].set_aspect('equal', adjustable='box')
axes[0].set_xlabel('$k_x$')
axes[0].set_ylabel('$k_y$')
axes[0].set_xlim(120, 136)
axes[0].set_ylim(120, 136)
fig.colorbar(im, ax=axes[0], label='difference')




d_enl = x_enl - xa_enl
mask_enl = np.abs(x_enl) < (np.mean(x_enl) + np.std(x_enl))
mask_enl = mask_enl.astype(float) 
v_enl = np.abs(xa_enl).max()
im_enl = axes[1].imshow(mask_enl, cmap='RdBu_r', vmin=-0, vmax=1)
axes[1].set_title('difference_enl')
axes[1].set_aspect('equal', adjustable='box')
axes[1].set_xlabel('$k_x$')
axes[1].set_ylabel('$k_y$')
axes[1].set_xlim(halfresizewid-legnth, halfresizewid+legnth)
axes[1].set_ylim(halfresizewid-legnth, halfresizewid+legnth)
fig.colorbar(im_enl, ax=axes[1], label='difference')

plt.tight_layout()
plt.show()

In [ ]:
im_grid0 = sp.ifft(cartesian_zte, axes=(-2, -1))
preloop = np.sum(np.abs(im_grid0)**2, axis=0)**0.5

zte_radial_kspace1, zte_radial_coords1 = asm.generate_zte_data_OLD(ksp=ksp, coord=coord, n_missing=0, undersampling_factor=1)
cartesian_nogap = asm.nufft_gridding(kspace=zte_radial_kspace1, coords=zte_radial_coords1)

dcf1 = (zte_radial_coords1[...,0]**2 + zte_radial_coords1[...,1]**2)**0.5
im_grid3 = sp.nufft_adjoint(zte_radial_kspace1* dcf1, zte_radial_coords1)
img_nogap = np.sum(np.abs(im_grid3)**2, axis=0)**0.5

In [ ]:
cy, cx = inner_wid // 2, inner_wid // 2
r = 5
yy, xx = np.ogrid[:inner_wid, :inner_wid]
maskb = (yy - cy)**2 + (xx - cx)**2 > r**2 +2

maskb[(cx-5):(cx+6), cy] = False
maskb[(cx), (cy-5):(cy+6)] = False

inner_removedb = inner_region.copy()
inner_removedb[:,~maskb] = 0

In [ ]:
masks = {'maskb': maskb}
xvals = [3,4,5]
for x in xvals:
    mask_enl = (np.abs(x_enl) < (np.mean(x_enl) + x*np.std(x_enl))).astype(float)
    masks[f'mask{x}'] = mask_enl
    outside = mask_enl == 0
    outside[35:65, 35:65] = False
    print(f'{x} times std is {outside.sum()}')

In [ ]:
cartlen = 20
radlen = 10

asm.plot_mask(inner_region, mask=masks['maskb'], zte_radial_coords=zte_radial_coordszero, zte_radial_kspace=zte_radial_kspacezero, innersidelen=inner_wid,sidelencart=cartlen,sidelenrad=radlen)
asm.plot_mask(inner_region, mask=masks['mask3'], zte_radial_coords=zte_radial_coordszero, zte_radial_kspace=zte_radial_kspacezero, innersidelen=inner_wid,sidelencart=cartlen,sidelenrad=radlen)
asm.plot_mask(inner_region, mask=masks['mask4'], zte_radial_coords=zte_radial_coordszero, zte_radial_kspace=zte_radial_kspacezero, innersidelen=inner_wid,sidelencart=cartlen,sidelenrad=radlen)
asm.plot_mask(inner_region, mask=masks['mask5'], zte_radial_coords=zte_radial_coordszero, zte_radial_kspace=zte_radial_kspacezero, innersidelen=inner_wid,sidelencart=cartlen,sidelenrad=radlen)

In [ ]:
def softimpute_ALS(X_H, M_H, rank, lamda, n_iters):
        I = np.eye(rank)
        m,n = np.shape(X_H)
        U = np.random.randn(m, rank) + 1j * np.random.randn(m, rank)
        V = np.random.randn(n, rank) + 1j * np.random.randn(n, rank)

        D = I.copy()


        A = np.dot(U,D)
        B = np.dot(V,D)
        iter_count = 0
        ABt = A @ B.conj().T
        
        while iter_count < n_iters:
            X_star = np.where(M_H, X_H, ABt)
            X_star1H = X_star.copy()
            A = X_star @ B @ np.linalg.inv(B.conj().T @ B + lamda*I)
            ABt = A @ B.conj().T
            X_star = np.where(M_H, X_H, ABt)
            B = X_star.conj().T @ A @ np.linalg.inv(A.conj().T @ A + lamda*I)
            ABt = A @ B.conj().T
            iter_count += 1
        return(ABt, X_star1H)

def LORAKS_imputeals(n_iters, window_size, cartesian_inputkspace, dtg_mask, rank, lamda, stride, im_dim):
    ksp_forhankel = cartesian_inputkspace.copy()
    ksp_zerod = ksp_forhankel * dtg_mask
    hankel_matrix, n_coils, Numx, Numy = asm.hankel_2(kspace=ksp_zerod, w=window_size, s=stride)
    mask_coiled = np.broadcast_to(dtg_mask, (cartesian_inputkspace.shape))
    masked_hankel, *_ = asm.hankel_2(kspace=mask_coiled, w=window_size, s=stride)
    masked_hankel = np.real(masked_hankel) > 0.5
    
    
    filled_hankel, X_star1 = softimpute_ALS(X_H = hankel_matrix, M_H = masked_hankel, rank = rank, lamda = lamda, n_iters=n_iters)

    kspace_cart_coils_recon = asm.hankel_H_averaged_2(filled_hankel, n_coils=ksp_forhankel.shape[0], Nx=ksp_forhankel.shape[1], Ny=ksp_forhankel.shape[2], w=window_size, s=stride)
    kspace_cart_coils_recon = np.where(dtg_mask, cartesian_inputkspace, kspace_cart_coils_recon)
    output_kspace = kspace_cart_coils_recon.copy()
    filled_ksp = rebuild(output_kspace=output_kspace, inner_mask = inner_mask, inner_start = start, inner_end = end, enlarged_kspace=cartesian_kspace_enl, resize_x = im_dim, resize_y = im_dim)
    im_grid0 = sp.ifft(filled_ksp, axes=(-2, -1))
    im_0 = np.sum(np.abs(im_grid0)**2, axis=0)**0.5

    return (im_0, filled_ksp)


def rebuild(output_kspace, inner_mask, inner_start, inner_end, enlarged_kspace, resize_x, resize_y):
    recombined = asm.jigsaw(output_kspace=output_kspace, isolation_mask = inner_mask, start=inner_start, end=inner_end, enlarged_kspace=enlarged_kspace)
    filled_ksp = asm.zero_padding(recombined, resize_x, resize_y)
    return(filled_ksp)

In [ ]:
window_size = 10
ksp_forhankel = inner_region.copy()
num_iters = 50


results = {}
results['preloop'] = {'im': preloop} 
for name, mask in masks.items():
    im_als, output_kspace = LORAKS_imputeals(n_iters=num_iters, window_size=window_size, cartesian_inputkspace=inner_region, dtg_mask=mask, rank= 10, lamda = 5*10**-5, stride=1, im_dim=256)
    results[name] = {'im': im_als}

results['nogap'] = {'im': img_nogap} 

ims = {f'rank_val {r}': results[r]['im'] for r in results}
asm.diff_matrix(ims)